# 07. Explainability
**목적**: 전체 모델의 global explanation과 개별 설비의 local explanation을 생성한다.

- **Case A**: SHAP summary plot + waterfall plot (per facility)
- **Case B**: Score decomposition — 각 component의 기여도 시각화

핵심 메시지:
> 고위험 설비가 *왜* 고위험인지를 구성요소별로 설명한다.

In [ ]:
import sys
sys.path.append('..')

import json, pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'AppleGothic'   # macOS 한글 폰트
matplotlib.rcParams['axes.unicode_minus'] = False
import warnings
warnings.filterwarnings('ignore')

from config import DATA_PROCESSED, OUT_FIGURES, OUT_TABLES

with open(DATA_PROCESSED / 'column_map.json', encoding='utf-8') as f:
    cmap = json.load(f)
FLAGS  = cmap['flags']
FC     = cmap['facility']
FID_COL = FC['facility_id']

df = pd.read_parquet(DATA_PROCESSED / 'risk_scores.parquet')
OUT_FIGURES.mkdir(parents=True, exist_ok=True)

---
## Case A — SHAP Explanation (label 있을 때)

In [ ]:
if FLAGS['has_label']:
    import shap

    with open(DATA_PROCESSED / 'best_model.pkl', 'rb') as f:
        saved = pickle.load(f)
    model = saved['model']
    FEATURE_COLS = saved['feature_cols']
    model_name = saved['name']

    df_weather = pd.read_parquet(DATA_PROCESSED / 'weather_features.parquet')
    df_buf     = pd.read_parquet(DATA_PROCESSED / 'buffer_features.parquet')
    buf_cols = [c for c in df_buf.columns if c != FID_COL]

    df_feat = df[[FID_COL, 'date', 'weather_hazard', 'spatial_exposure',
                  'facility_exposure', 'hist_prior']].copy()
    df_feat = df_feat.merge(df_weather[[FID_COL, 'date'] + FEATURE_COLS], on=[FID_COL, 'date'], how='left')
    df_feat = df_feat.merge(df_buf[[FID_COL] + buf_cols], on=FID_COL, how='left')

    X = df_feat[FEATURE_COLS].fillna(df_feat[FEATURE_COLS].median())

    # ── 모델 유형별 올바른 Explainer 선택 ──────────────────────────────────
    if model_name in ['lightgbm', 'xgboost', 'random_forest']:
        # 트리 기반 모델 → TreeExplainer (빠르고 정확)
        explainer = shap.TreeExplainer(model)
        print(f'✓ TreeExplainer 사용: {model_name}')
    elif model_name == 'logistic':
        # 선형 모델 → LinearExplainer
        explainer = shap.LinearExplainer(model, X)
        print(f'✓ LinearExplainer 사용: {model_name}')
    else:
        # Fallback → KernelExplainer (느리지만 범용)
        background = shap.sample(X, 100, random_state=42)
        explainer = shap.KernelExplainer(model.predict_proba, background)
        print(f'✓ KernelExplainer 사용 (Fallback): {model_name}')

    print(f'모델: {model_name}')

In [ ]:
if FLAGS['has_label']:
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X)

    # TreeExplainer 반환 형식에 따라 처리
    if isinstance(shap_values, list):
        sv = shap_values[1]   # class=1 (위험)
    else:
        sv = shap_values

    print(f'SHAP values shape: {sv.shape}')

In [ ]:
if FLAGS['has_label']:
    # Global Explanation — SHAP Summary Plot
    fig, ax = plt.subplots(figsize=(10, 8))
    shap.summary_plot(sv, X, max_display=15, show=False)
    plt.title('Global Feature Importance (SHAP)', fontsize=14)
    plt.tight_layout()
    plt.savefig(OUT_FIGURES / 'shap_summary.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('shap_summary.png 저장')

In [ ]:
if FLAGS['has_label']:
    # Local Explanation — Top-5 고위험 설비별 Waterfall Plot
    top5_idx = df['final_risk'].nlargest(5).index

    for i, row_idx in enumerate(top5_idx):
        fid = df.loc[row_idx, FID_COL]
        risk = df.loc[row_idx, 'final_risk']

        if row_idx < len(sv):
            shap_exp = shap.Explanation(
                values=sv[row_idx],
                base_values=explainer.expected_value[1] if isinstance(explainer.expected_value, list) else explainer.expected_value,
                data=X.iloc[row_idx].values,
                feature_names=FEATURE_COLS
            )
            fig = plt.figure(figsize=(10, 5))
            shap.waterfall_plot(shap_exp, max_display=10, show=False)
            plt.title(f'Local Explanation — {fid}  (Risk: {risk:.1f})', fontsize=12)
            plt.tight_layout()
            plt.savefig(OUT_FIGURES / f'local_shap_{i+1}_{fid}.png', dpi=150, bbox_inches='tight')
            plt.close()
            print(f'  {fid}: {risk:.1f}점 — waterfall 저장')

---
## Case B — Score Decomposition + KernelExplainer SHAP (label 없을 때)

**두 가지 방법을 모두 제공한다:**
1. **Score Decomposition**: 빠르고 직관적 (가중치 기반 기여도)
2. **KernelExplainer SHAP**: 수학적으로 엄밀한 Shapley value (rule-based에도 적용 가능)

In [ ]:
if not FLAGS['has_label']:
    # Global Explanation — 구성요소별 평균 기여도
    from config import WEIGHTS
    w = WEIGHTS['base']

    df['weather_contrib']   = w['weather']  * df['weather_hazard'].fillna(50)
    df['spatial_contrib']   = w['spatial']  * df['spatial_exposure'].fillna(50)
    df['facility_contrib']  = w['facility'] * df['facility_exposure'].fillna(50)
    df['prior_contrib']     = w['prior']    * df['hist_prior'].fillna(0)

    component_cols = ['weather_contrib', 'spatial_contrib', 'facility_contrib', 'prior_contrib']
    labels = ['Weather\nHazard', 'Spatial\nExposure', 'Facility\nExposure', 'Historical\nPrior']
    colors = ['#d62728', '#ff7f0e', '#1f77b4', '#2ca02c']

    means = [df[c].mean() for c in component_cols]

    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.bar(labels, means, color=colors, edgecolor='white', linewidth=1.5)
    for bar, val in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                f'{val:.1f}', ha='center', va='bottom', fontsize=11)
    ax.set_ylabel('평균 기여도 점수', fontsize=12)
    ax.set_title('Global Component Contribution (전체 설비 평균)', fontsize=13)
    ax.set_ylim(0, max(means) * 1.2)
    plt.tight_layout()
    plt.savefig(OUT_FIGURES / 'global_contribution.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('global_contribution.png 저장')

In [ ]:
if not FLAGS['has_label']:
    # Local Explanation — Top-10 고위험 설비 기여도 테이블
    latest = df.sort_values('date').groupby(FID_COL).last().reset_index()
    top10 = latest.nlargest(10, 'final_risk')

    explanation_rows = []
    for _, row in top10.iterrows():
        contribs = {
            FID_COL: row[FID_COL],
            'final_risk': round(row['final_risk'], 1),
            'risk_grade': row['risk_grade'],
            'weather_contrib':  round(row['weather_contrib'],  1),
            'spatial_contrib':  round(row['spatial_contrib'],  1),
            'facility_contrib': round(row['facility_contrib'], 1),
            'prior_contrib':    round(row['prior_contrib'],    1),
        }
        # 주요 원인 1~3위
        sorted_c = sorted(
            [('기상위험', contribs['weather_contrib']),
             ('공간노출', contribs['spatial_contrib']),
             ('설비노출', contribs['facility_contrib']),
             ('과거화재', contribs['prior_contrib'])],
            key=lambda x: -x[1]
        )
        contribs['main_reason'] = ' / '.join(f'{n}({v:.0f})' for n, v in sorted_c[:2])
        explanation_rows.append(contribs)

    df_explain = pd.DataFrame(explanation_rows)
    df_explain.to_csv(OUT_TABLES / 'top10_explanation.csv', index=False)
    print(df_explain.to_string(index=False))

In [ ]:
if not FLAGS['has_label']:
    # Local Explanation — Top-5 설비 Stacked Bar
    top5 = df_explain.head(5)

    fig, ax = plt.subplots(figsize=(10, 5))
    fids = top5[FID_COL].astype(str)
    b1 = ax.bar(fids, top5['weather_contrib'],  label='Weather Hazard',   color='#d62728')
    b2 = ax.bar(fids, top5['spatial_contrib'],  bottom=top5['weather_contrib'],
                label='Spatial Exposure', color='#ff7f0e')
    b3 = ax.bar(fids, top5['facility_contrib'], bottom=top5['weather_contrib']+top5['spatial_contrib'],
                label='Facility Exposure', color='#1f77b4')
    b4 = ax.bar(fids, top5['prior_contrib'],
                bottom=top5['weather_contrib']+top5['spatial_contrib']+top5['facility_contrib'],
                label='Historical Prior', color='#2ca02c')

    for i, (_, row) in enumerate(top5.iterrows()):
        ax.text(i, row['final_risk'] + 0.5, f"{row['final_risk']:.0f}",
                ha='center', fontsize=10, fontweight='bold')

    ax.set_ylabel('위험도 점수 (0~100)', fontsize=12)
    ax.set_title('Top-5 고위험 설비 — 위험 구성요소 분해', fontsize=13)
    ax.legend(loc='upper right')
    ax.set_ylim(0, 110)
    plt.tight_layout()
    plt.savefig(OUT_FIGURES / 'local_decomposition_top5.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('local_decomposition_top5.png 저장')

print('\n다음 단계: 08_validation.ipynb')

### Case B-2. KernelExplainer SHAP (Rule-based 모델에 적용)

In [ ]:
if not FLAGS['has_label']:
    import shap
    from config import WEIGHTS

    # Rule-based risk function을 SHAP이 분석할 수 있는 형태로 래핑
    df_buf_filled = pd.read_parquet(DATA_PROCESSED / 'buffer_features.parquet')
    df_weather_feat = pd.read_parquet(DATA_PROCESSED / 'weather_features.parquet')

    # 최신 날짜 기준 설비별 feature matrix 구성
    latest_date = df['date'].max()
    df_latest = df[df['date'] == latest_date].copy()

    feature_cols_b = ['weather_hazard', 'spatial_exposure', 'facility_exposure', 'hist_prior']
    X_b = df_latest[feature_cols_b].fillna(df_latest[feature_cols_b].median()).values

    w = WEIGHTS['base']
    def risk_fn(X):
        """가중합 rule-based risk function"""
        return (
            w['weather']  * X[:, 0] +
            w['spatial']  * X[:, 1] +
            w['facility'] * X[:, 2] +
            w['prior']    * X[:, 3]
        )

    # 배경 데이터로 100개 샘플
    np.random.seed(42)
    bg_idx = np.random.choice(len(X_b), min(100, len(X_b)), replace=False)
    background = X_b[bg_idx]

    explainer_b = shap.KernelExplainer(risk_fn, background)

    # Top-20 고위험 설비에 대해서만 계산 (속도)
    top20_mask = df_latest['final_risk'].nlargest(20).index
    X_top20 = X_b[df_latest.index.get_indexer(top20_mask)]

    print('KernelExplainer SHAP 계산 중... (Top-20 설비, 약 30초)')
    shap_vals_b = explainer_b.shap_values(X_top20, silent=True)

    # SHAP Summary Plot (4개 feature)
    fig, ax = plt.subplots(figsize=(8, 4))
    shap.summary_plot(
        shap_vals_b,
        X_top20,
        feature_names=feature_cols_b,
        plot_type='bar',
        show=False
    )
    plt.title('SHAP Feature Importance (KernelExplainer, Top-20 설비)', fontsize=13)
    plt.tight_layout()
    plt.savefig(OUT_FIGURES / 'shap_kernel_summary.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('shap_kernel_summary.png 저장')

    # 검증: SHAP 합 ≈ 예측값 - 기준값
    baseline = risk_fn(background).mean()
    pred_top1 = risk_fn(X_top20[:1])[0]
    shap_sum  = shap_vals_b[0].sum()
    print(f'\n[SHAP 정합성 검증]')
    print(f'  기준값(baseline):  {baseline:.2f}')
    print(f'  Top-1 예측값:      {pred_top1:.2f}')
    print(f'  SHAP 합계:         {shap_sum:.2f}')
    print(f'  baseline + SHAP:   {baseline + shap_sum:.2f}  ← 예측값과 일치해야 함')

    # 결과 저장
    import pickle as pkl
    with open(DATA_PROCESSED / 'shap_kernel_results.pkl', 'wb') as f:
        pkl.dump({
            'shap_values': shap_vals_b,
            'feature_cols': feature_cols_b,
            'baseline': baseline,
            'facility_ids': df_latest.loc[top20_mask, FID_COL].values,
            'final_risks': df_latest.loc[top20_mask, 'final_risk'].values,
        }, f)
    print('\nshap_kernel_results.pkl 저장 완료')